In [ ]:
import requests

url      = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(url)

with open("shakespeare.txt", "w") as f:
    f.write(response.text)

In [ ]:
with open("shakespeare.txt", "r") as f:
    text = f.read()

In [ ]:
import re
def tokenize(text):
  text= text.replace("?","")
  text= text.lower()
  text = re.sub(r"[^a-z\s]", "", text)
  return text.split()

In [ ]:
tokenize(text)

['first',
 'citizen',
 'before',
 'we',
 'proceed',
 'any',
 'further',
 'hear',
 'me',
 'speak',
 'all',
 'speak',
 'speak',
 'first',
 'citizen',
 'you',
 'are',
 'all',
 'resolved',
 'rather',
 'to',
 'die',
 'than',
 'to',
 'famish',
 'all',
 'resolved',
 'resolved',
 'first',
 'citizen',
 'first',
 'you',
 'know',
 'caius',
 'marcius',
 'is',
 'chief',
 'enemy',
 'to',
 'the',
 'people',
 'all',
 'we',
 'knowt',
 'we',
 'knowt',
 'first',
 'citizen',
 'let',
 'us',
 'kill',
 'him',
 'and',
 'well',
 'have',
 'corn',
 'at',
 'our',
 'own',
 'price',
 'ist',
 'a',
 'verdict',
 'all',
 'no',
 'more',
 'talking',
 'ont',
 'let',
 'it',
 'be',
 'done',
 'away',
 'away',
 'second',
 'citizen',
 'one',
 'word',
 'good',
 'citizens',
 'first',
 'citizen',
 'we',
 'are',
 'accounted',
 'poor',
 'citizens',
 'the',
 'patricians',
 'good',
 'what',
 'authority',
 'surfeits',
 'on',
 'would',
 'relieve',
 'us',
 'if',
 'they',
 'would',
 'yield',
 'us',
 'but',
 'the',
 'superfluity',
 'while

In [ ]:
vocab = {"<UNK>": 0}
for word in tokenize(text):
  if word  not in vocab:
    vocab[word]= len(vocab)


In [ ]:
vocab.keys()

dict_keys(['<UNK>', 'first', 'citizen', 'before', 'we', 'proceed', 'any', 'further', 'hear', 'me', 'speak', 'all', 'you', 'are', 'resolved', 'rather', 'to', 'die', 'than', 'famish', 'know', 'caius', 'marcius', 'is', 'chief', 'enemy', 'the', 'people', 'knowt', 'let', 'us', 'kill', 'him', 'and', 'well', 'have', 'corn', 'at', 'our', 'own', 'price', 'ist', 'a', 'verdict', 'no', 'more', 'talking', 'ont', 'it', 'be', 'done', 'away', 'second', 'one', 'word', 'good', 'citizens', 'accounted', 'poor', 'patricians', 'what', 'authority', 'surfeits', 'on', 'would', 'relieve', 'if', 'they', 'yield', 'but', 'superfluity', 'while', 'were', 'wholesome', 'might', 'guess', 'relieved', 'humanely', 'think', 'too', 'dear', 'leanness', 'that', 'afflicts', 'object', 'of', 'misery', 'as', 'an', 'inventory', 'particularise', 'their', 'abundance', 'sufferance', 'gain', 'them', 'revenge', 'this', 'with', 'pikes', 'ere', 'become', 'rakes', 'for', 'gods', 'i', 'in', 'hunger', 'bread', 'not', 'thirst', 'especially',

In [ ]:
list(vocab.items())[1]

('first', 1)

In [ ]:
def text_to_index(text , vocab):
    index_text = []
    for word in tokenize(text):
        if word in vocab:
            index_text.append(vocab[word])
        else:
            index_text.append(vocab["<UNK>"])
            # return [vocab.get(word, 0) for word in tokenize(text)]
    return index_text

In [ ]:
idx=text_to_index(text , vocab)

In [ ]:
# sequences
inputs = []
targets = []
seq_len= 10
for i in range(len(idx)-seq_len):
  inputs.append(idx[i:i+seq_len])
  targets.append(idx[i+seq_len])


In [ ]:
# creating the dataset
from torch.utils.data import Dataset , DataLoader
import torch
class word_pred_dataset(Dataset):
  def __init__(self, inputs, targets):
    self.inputs= inputs
    self.targets= targets
  def __len__(self):
    return len(self.inputs)
  def __getitem__(self, index):
     x= torch.tensor(self.inputs[index] , dtype= torch.long)
     y= torch.tensor(self.targets[index] , dtype= torch.long)
     return x, y


In [ ]:
dataset = word_pred_dataset(inputs, targets)

In [ ]:
print(f"Total samples : {len(dataset)}")

x, y = dataset[20]
print(f"Input tensor  : {x}")
print(f"Target tensor : {y}")

Total samples : 202609
Input tensor  : tensor([16, 17, 18, 16, 19, 11, 14, 14,  1,  2])
Target tensor : 1


In [ ]:
# Daraloader
data_loader = DataLoader(dataset=dataset , batch_size=32 , shuffle= True)

In [ ]:
len(data_loader)

6332

In [ ]:
len(vocab)

12848

In [ ]:
# bulding the model arctecture
from torch import nn

class pred_word_model(nn.Module):
  def __init__(self):
    super().__init__()
    self.embeeding = nn.Embedding(len(vocab) , 128 , padding_idx=0)
    self.lstm1 = nn.LSTM(128, 256, num_layers=1,
                             batch_first=True)

    self.drop = nn.Dropout(0.3)
    self.fc= nn.Sequential(
        nn.Linear(256, 512),
        nn.ReLU(),
        nn.Linear(512, len(vocab))

    )
  def forward(self,x ):
    embeeded = self.embeeding(x)
    output1 , (hidden1 , cell1) = self.lstm1(embeeded)
    out = output1[:, -1, :]
    out = self.drop(out)
    out = self.fc(out)
    return out



In [ ]:
test_input = torch.randint(0, len(vocab), (1, 10))
print(f"Input shape  : {test_input.shape}")
model = pred_word_model()
output = model(test_input)
print(f"Output shape : {output.shape}")


Input shape  : torch.Size([1, 10])
Output shape : torch.Size([1, 12848])


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
model = pred_word_model().to(device)

In [ ]:
!pip install torchinfo


In [ ]:
from torchinfo import summary
import torch
summary(model , input_size=(32, 10), dtypes=[torch.long])

Layer (type:depth-idx)                   Output Shape              Param #
pred_word_model                          [32, 12848]               --
├─Embedding: 1-1                         [32, 10, 128]             1,644,544
├─LSTM: 1-2                              [32, 10, 256]             395,264
├─Dropout: 1-3                           [32, 256]                 --
├─Sequential: 1-4                        [32, 12848]               --
│    └─Linear: 2-1                       [32, 512]                 131,584
│    └─ReLU: 2-2                         [32, 512]                 --
│    └─Linear: 2-3                       [32, 12848]               6,591,024
Total params: 8,762,416
Trainable params: 8,762,416
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 394.23
Input size (MB): 0.00
Forward/backward pass size (MB): 4.40
Params size (MB): 35.05
Estimated Total Size (MB): 39.46

In [ ]:
# loss + optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters() , lr=0.001)

In [ ]:
from tqdm.auto import tqdm
from timeit import default_timer as timer
from sklearn.metrics import accuracy_score

In [ ]:
# training the model
epoches  = 50
start = timer()
for epoch in tqdm(range(epoches)):
  model.train()
  train_loss , train_acc = 0 , 0
  for X , y in data_loader:
    X= X.to(device)
    y= y.to(device)
    optimizer.zero_grad()
    y_pred = model(X)
    loss = loss_fn(y_pred , y)
    train_loss += loss.item()
    acc = accuracy_score(y.cpu().numpy() , torch.argmax(y_pred , dim=1) .cpu().numpy())
    train_acc += acc
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
  train_loss = train_loss / len(data_loader)
  train_acc = train_acc / len(data_loader)
  end = timer()

  print(f"Epoch {epoch + 1 } | train_loss {train_loss :.4f} | train_accuracy {train_acc:.4f}")
  print(f"Time taken : {end - start}")

  0%|          | 0/50 [00:00<?, ?it/s]

Epoch 1 | train_loss 6.5602 | train_accuracy 0.0653
Time taken : 50.358403450000026
Epoch 2 | train_loss 6.0770 | train_accuracy 0.0835
Time taken : 91.38804761600005
Epoch 3 | train_loss 5.7957 | train_accuracy 0.0901
Time taken : 133.19923671000004
Epoch 4 | train_loss 5.6233 | train_accuracy 0.0944
Time taken : 175.00286507600003
Epoch 5 | train_loss 5.4993 | train_accuracy 0.0978
Time taken : 216.51148155200008
Epoch 6 | train_loss 5.4071 | train_accuracy 0.1004
Time taken : 257.85354668
Epoch 7 | train_loss 5.3334 | train_accuracy 0.1029
Time taken : 298.804750032
Epoch 8 | train_loss 5.2708 | train_accuracy 0.1050
Time taken : 340.131619161
Epoch 9 | train_loss 5.2161 | train_accuracy 0.1068
Time taken : 381.908480059
Epoch 10 | train_loss 5.1693 | train_accuracy 0.1091
Time taken : 423.02519968
Epoch 11 | train_loss 5.1320 | train_accuracy 0.1112
Time taken : 464.01357320700004
Epoch 12 | train_loss 5.0935 | train_accuracy 0.1122
Time taken : 505.650895999
Epoch 13 | train_loss 

In [ ]:
torch.save(model.state_dict(), "next_word_model.pth")
print("Model saved ")

Model saved 


In [ ]:
from google.colab import files
files.download("next_word_model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

def generate_text(model, start_text, vocab, seq_len=10, gen_len=50, device="cpu"):

    model.eval()
    idx_to_word = {v: k for k, v in vocab.items()}
    words = start_text.lower().split()
    input_seq = [vocab.get(word, vocab["<UNK>"]) for word in words]
    generated = input_seq.copy()
    for _ in range(gen_len):
        x = torch.tensor([generated[-seq_len:]], dtype=torch.long).to(device)
        with torch.no_grad():
            y_pred = model(x)
            next_idx = torch.argmax(y_pred, dim=1).item()
        generated.append(next_idx)
    generated_text = " ".join([idx_to_word[i] for i in generated])
    return generated_text

In [ ]:

start_text = "what will the "
gen_len = 10
seq_len = 10
generated = generate_text(model, start_text, vocab, seq_len=seq_len, gen_len=gen_len, device=device)
print(generated)

what will the king and the rest stand up duchess of york ah
